# 10 — Executive Summary

Visual narrative for decision-makers. One chart per key finding, connecting the story
from RQ1 (volume drivers) through RQ6 (resolution speed).

This is **synthesis**, not analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, load_metrics, METRICS_DIR,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()

# Load all metrics from previous notebooks
m = {}
for name in ["02_general_overview", "03_volume_drivers", "04_hospital_efficiency",
             "05_financial_analysis", "06_bed_savings", "07_mortality_outcomes",
             "08_resolution_speed", "09_ml_models"]:
    try:
        m[name] = load_metrics(name)
    except FileNotFoundError:
        print(f"WARNING: {name} metrics not found")
        m[name] = {}

print(f"Loaded metrics from {len([k for k in m if m[k]])} notebooks")

Loaded metrics from 8 notebooks


## The Big Numbers

In [2]:
overview = m.get("02_general_overview", {})
financial = m.get("05_financial_analysis", {})
mortality = m.get("07_mortality_outcomes", {})
beds = m.get("06_bed_savings", {})

big_numbers = [
    ("Admissions", f"{overview.get('total_admissions', 0):,}", "2016-2025"),
    ("Hospitals", f"{overview.get('total_hospitals', 0):,}", "treating N20"),
    ("Total Cost", f"R${overview.get('total_cost_brl', 0)/1e6:.0f}M", "SUS billing"),
    ("Mean LOS", f"{overview.get('mean_los', 0):.1f}d", f"median {overview.get('median_los', 0):.0f}d"),
    ("Mortality", f"{overview.get('mortality_rate', 0):.2f}%", f"{overview.get('total_deaths', 0):,} deaths"),
    ("Excess Cost", f"R${financial.get('excess_cost_brl', 0)/1e6:.1f}M",
     f"{financial.get('excess_cost_pct', 0):.0f}% of total"),
]

fig, axes = plt.subplots(1, len(big_numbers), figsize=(20, 3))
colors = ["#2563EB", "#7C3AED", "#059669", "#D97706", "#DC2626", "#6B7280"]

for ax, (title, value, subtitle), color in zip(axes, big_numbers, colors):
    ax.text(0.5, 0.7, value, ha="center", va="center", fontsize=24, fontweight="bold", color=color,
            transform=ax.transAxes)
    ax.text(0.5, 0.35, title, ha="center", va="center", fontsize=12, fontweight="bold",
            transform=ax.transAxes)
    ax.text(0.5, 0.15, subtitle, ha="center", va="center", fontsize=9, color="gray",
            transform=ax.transAxes)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

fig.suptitle("Kidney Stone Hospitalizations — São Paulo State",
             fontsize=16, fontweight="bold", y=1.05)
fig.tight_layout()
save_plot(fig, "big_numbers", prefix="10")

  Saved: 10_big_numbers.png


## Finding 1: Hospital Matters More Than Procedure (RQ2)

In [3]:
eff = m.get("04_hospital_efficiency", {})

fig, ax = plt.subplots(figsize=(10, 5))
effects = {
    f"Hospital effect\n({eff.get('hospital_effect_std', 0):.1f}d std)": eff.get("hospital_effect_std", 0),
    f"Procedure effect\n({eff.get('procedure_effect_std', 0):.1f}d std)": eff.get("procedure_effect_std", 0),
}
ax.bar(effects.keys(), effects.values(), color=["#DC2626", "#2563EB"], width=0.5)
ax.set_title(f"Hospital Effect Is {eff.get('effect_ratio', 0)}x Larger Than Procedure Effect",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Standard Deviation of Mean LOS (days)")
fig.tight_layout()
save_plot(fig, "hospital_vs_procedure", prefix="10")

  Saved: 10_hospital_vs_procedure.png


## Finding 2: The Three Levers for Bed Savings (RQ4)

In [4]:
total_bd = beds.get("total_bed_days", 1)

levers = {
    "Long-stay\nreduction": beds.get("lever1_long_stay_savings", 0),
    "Diagnostic\noutpatient shift": beds.get("lever2_diagnostic_shift_savings", 0),
    "Hospital\nstandardization": beds.get("lever3_standardization_savings", 0),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2563EB", "#D97706", "#059669"]
bars = ax.bar(levers.keys(), levers.values(), color=colors)
for bar, v in zip(bars, levers.values()):
    ax.text(bar.get_x() + bar.get_width()/2, v + total_bd * 0.005,
            f"{v:,.0f}\n({v/total_bd*100:.1f}%)", ha="center", fontsize=10)
ax.set_title("Bed-Day Savings by Lever", fontsize=14, fontweight="bold")
ax.set_ylabel("Bed-days saved")
fig.tight_layout()
save_plot(fig, "three_levers", prefix="10")

  Saved: 10_three_levers.png


## Finding 3: Long Stay = Higher Mortality (RQ5)

In [5]:
mort = m.get("07_mortality_outcomes", {})

fig, ax = plt.subplots(figsize=(10, 5))
labels = ["Short stay (<=7d)", "Long stay (>7d)"]
rates = [mort.get("short_stay_mortality_rate", 0), mort.get("long_stay_mortality_rate", 0)]
bars = ax.bar(labels, rates, color=["#059669", "#DC2626"], width=0.5)
for bar, v in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}%", ha="center", fontsize=12)

ratio = mort.get("mortality_ratio_long_vs_short", 0)
ax.set_title(f"Long-Stay Mortality Is {ratio:.0f}x Higher\n"
             f"(estimated {mort.get('lives_saveable_estimate', 0)} lives saveable)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Mortality Rate (%)")
fig.tight_layout()
save_plot(fig, "mortality_gradient", prefix="10")

  Saved: 10_mortality_gradient.png


## Finding 4: Ureteroscopy Adoption Drives Volume (RQ1)

In [6]:
vol = m.get("03_volume_drivers", {})

yearly = kidney.groupby("year").agg(
    total=pd.NamedAgg(column="year", aggfunc="count"),
    ureteroscopy=pd.NamedAgg(column="has_new_proc", aggfunc="sum"),
)
yearly["legacy"] = yearly["total"] - yearly["ureteroscopy"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly.index, yearly["legacy"], label="Legacy procedures", color="#6B7280")
ax.bar(yearly.index, yearly["ureteroscopy"], bottom=yearly["legacy"],
       label="Ureteroscopy (modern)", color="#7C3AED")
ax.set_title(f"Volume Growth Decomposition\n"
             f"(Ureteroscopy: {vol.get('ureteroscopy_latest_year_share', 0):.0f}% of latest year)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Admissions")
ax.legend()
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_plot(fig, "volume_decomposition", prefix="10")

  Saved: 10_volume_decomposition.png


## Finding 5: The Diagnostic Admission Problem (RQ3, RQ6)

In [7]:
res = m.get("08_resolution_speed", {})
fin = m.get("05_financial_analysis", {})

fig, ax = plt.subplots(figsize=(10, 5))
data = {
    f"Diagnostic admissions\n({res.get('diagnostic_pct', 0):.0f}% of all)": res.get("diagnostic_pct", 0),
    f"Hospitals >50% diagnostic\n({res.get('hospitals_gt50pct_diagnostic', 0)})": (
        res.get("hospitals_gt50pct_diagnostic", 0) / max(overview.get("total_hospitals", 1), 1) * 100
    ),
    f"Migration rate\n({res.get('migration_rate', 0):.0f}%)": res.get("migration_rate", 0),
}
ax.bar(data.keys(), data.values(), color=["#DC2626", "#D97706", "#7C3AED"])
ax.set_title("Key Resolution Bottlenecks", fontsize=14, fontweight="bold")
ax.set_ylabel("%")
fig.tight_layout()
save_plot(fig, "bottlenecks", prefix="10")

  Saved: 10_bottlenecks.png


## Summary Dashboard

In [8]:
print("="*60)
print("  EXECUTIVE SUMMARY")
print("  Kidney Stone Hospitalizations in São Paulo (SUS)")
print("="*60)
print(f"\n  Total admissions:    {overview.get('total_admissions', 0):>10,}")
print(f"  Hospitals:           {overview.get('total_hospitals', 0):>10,}")
print(f"  Total cost:          R${overview.get('total_cost_brl', 0)/1e6:>8.1f}M")
print(f"  Excess cost:         R${financial.get('excess_cost_brl', 0)/1e6:>8.1f}M ({financial.get('excess_cost_pct', 0):.0f}%)")
print(f"  Total deaths:        {overview.get('total_deaths', 0):>10,}")
print(f"  Lives saveable:      {mortality.get('lives_saveable_estimate', 0):>10,}")
print(f"\n  --- Key Findings ---")
print(f"  Hospital effect:     {eff.get('effect_ratio', 0)}x larger than procedure effect")
print(f"  Bed-days saveable:   {beds.get('combined_conservative', 0):>10,} (conservative)")
print(f"  Diagnostic problem:  {res.get('diagnostic_pct', 0):.0f}% of admissions are diagnostic")
print(f"  ML model R²:         {m.get('09_ml_models', {}).get('regression_r2_mean', 0):.3f}")
print("\n" + "="*60)

  EXECUTIVE SUMMARY
  Kidney Stone Hospitalizations in São Paulo (SUS)

  Total admissions:       206,500
  Hospitals:                  510
  Total cost:          R$   187.8M
  Excess cost:         R$    41.8M (22%)
  Total deaths:               714
  Lives saveable:             340

  --- Key Findings ---
  Hospital effect:     1.9x larger than procedure effect
  Bed-days saveable:      268,608 (conservative)
  Diagnostic problem:  20% of admissions are diagnostic
  ML model R²:         0.155

